# Project Name: Indonesian Municipal Fiscal Analytics Framework
## Module: 04_Finalizing

**Project Pipeline Status:**
- [x] **00_Data_Acquisition.ipynb** -> Documents the source publication and how the raw table data was extracted from BPS PDF reports.
- [x] **01_Data_Preprocessing.ipynb** -> Loads the raw BPS fiscal CSV and runs data-quality checks (shape, dtypes, missing values, duplicates).
- [x] **02_Data_Analysis.ipynb** -> Explores distribution shape/skewness of the 8 fiscal ratios and applies a log1p transform where it helps.
- [x] **03_Modelling.ipynb** -> Standardizes features, selects k, fits K-Means (k=4), and profiles/visualizes the resulting clusters.
- [ ] **04_Finalizing.ipynb** *(current)* -> Checks the geographic pattern of clusters, saves the final labeled dataset, and writes the project summary.

---
### 🎯 Module Objective
Sanity-check the K-Means clusters against real geography, save the final labeled dataset for downstream use, and summarize the end-to-end pipeline outcome and its explicit limitations.

### 📥 Data Ingestion
* **Source File:** `data/processed/fiscal_clustered_with_pca.csv`
* **Current Shape:** 508 rows × 22 columns (original + log + cluster + cluster_label + pc1/pc2)

### 🛠️ Environment Setup
```python
import pandas as pd

df = pd.read_csv('data/processed/fiscal_clustered_with_pca.csv')
cluster_labels = {
    0: 'Dependent / Underperforming',
    1: 'Resource-Windfall Overperformers',
    2: 'Fiscally Autonomous',
    3: 'Stable / Average',
}
```
---


In [ ]:
import pandas as pd

df = pd.read_csv('data/processed/fiscal_clustered_with_pca.csv')

cluster_labels = {
    0: 'Dependent / Underperforming',
    1: 'Resource-Windfall Overperformers',
    2: 'Fiscally Autonomous',
    3: 'Stable / Average',
}
print('Loaded shape:', df.shape)


### 7.5 Geographic pattern

Do the clusters line up with real geography? We check the top provinces represented in each cluster.

In [17]:
for label in cluster_labels.values():
    subset = df[df['cluster_label'] == label]
    print(f"--- {label} (n={len(subset)}) ---")
    print(subset['provinsi'].value_counts().head(5).to_string())
    print()

--- Dependent / Underperforming (n=74) ---
provinsi
Nusa Tenggara Timur    9
Maluku                 8
Maluku Utara           6
Lampung                4
Papua                  4

--- Resource-Windfall Overperformers (n=46) ---
provinsi
Kalimantan Selatan    8
Kalimantan Timur      7
Papua Tengah          5
Kalimantan Tengah     4
Kalimantan Utara      4

--- Fiscally Autonomous (n=156) ---
provinsi
Jawa Tengah    30
Jawa Timur     26
Jawa Barat     20
Bali            9
Banten          7

--- Stable / Average (n=232) ---
provinsi
Sumatera Utara       23
Aceh                 20
Sumatera Barat       15
Sulawesi Selatan     14
Sulawesi Tenggara    13



**Reading the pattern:**
- **Dependent / Underperforming** clusters in NTT, Maluku, Maluku Utara, Papua — the well-known eastern-Indonesia fiscal gap.
- **Resource-Windfall Overperformers** clusters in Kalimantan (Selatan/Timur/Tengah/Utara) and Papua Tengah — mining/resource royalty regions whose modest PAD targets get blown past.
- **Fiscally Autonomous** clusters in Jawa Tengah/Timur/Barat, Bali, Banten — the industrial/tourism belt.
- **Stable / Average** spreads across Sumatera and Sulawesi provinces — the largest, most "typical" group.

**Caveat worth stating plainly in any writeup:** silhouette scores were modest (~0.21), meaning these are soft, overlapping groupings, not hard-edged categories. That's an accurate reflection of real fiscal data, not a weakness to hide.

In [18]:
df.to_csv("fiscal_clustered_indonesia_2023.csv", index=False)
print("Saved: fiscal_clustered_indonesia_2023.csv")
print("Final shape:", df.shape)

Saved: fiscal_clustered_indonesia_2023.csv
Final shape: (508, 22)


---
### 📤 Data Export & Handoff
* **Output File:** `data/processed/fiscal_clustered_indonesia_2023.csv` *(final deliverable — no further notebook consumes this)*
* **Next Destination:** `README.md` (results matrix / summary section)


## Summary

| Step | Outcome |
|---|---|
| Load & clean data | 508 regions, 8 fiscal indicators, no missing values |
| Handle skew | `log1p` transform applied, worked well for 6/8 indicators (2 flagged honestly) |
| Cluster (k=4) | Modest silhouette (~0.21) — soft, overlapping groups, not hard boundaries |
| Profile clusters | Dependent/Underperforming, Resource-Windfall Overperformers, Fiscally Autonomous, Stable/Average |
| Geographic check | Clusters align with known regional patterns (eastern Indonesia dependency, Java/Bali autonomy, Kalimantan resource windfalls) |

**Not covered here** (deliberately out of scope): spatial autocorrelation (needs region boundary geometry, not available), VaR/CVaR-style risk modeling (not statistically supportable on single-year cross-sectional ratios), transfer allocation optimization (inputs not in the current data).